In [ ]:
import os, glob, time, pandas as pd, numpy as np
R=[p for p in glob.glob('/kaggle/input/**/train.csv',recursive=True) if 'knee' in p.lower() or True][0]
D=os.path.dirname(R); print(D, os.listdir(D))
tr=pd.read_csv(D+'/train.csv'); ts=pd.read_csv(D+'/train_series.csv')
L=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
print('train rows',len(tr), 'cols', list(tr.columns))
lab=tr[L].notna().all(1); print('fully labeled', lab.sum(), 'any label', tr[L].notna().any(1).sum())
print(tr.loc[lab,L].mean().round(3).to_dict())
print('series rows',len(ts)); print(ts.groupby(['Anatomical_Plane','Fluid_Sensitive','Fat_Suppression']).size())
print('series per study', ts.groupby('StudyInstanceUID').size().describe())
print(tr.Report.str.len().describe())
for r in tr.Report.sample(6,random_state=0): print('-----\n',r[:700])
print(pd.read_csv(D+'/test.csv').head()); print(pd.read_csv(D+'/test_series.csv').head(10)); print(pd.read_csv(D+'/sample_submission.csv').head(2).T)


In [ ]:
import pydicom
s=ts.iloc[0]; p=f"{D}/train_series/{s.StudyInstanceUID}/{s.SeriesInstanceUID}"
fs=sorted(os.listdir(p)); print(len(fs))
t=time.time(); n=0
for i,(st,se) in enumerate(ts[['StudyInstanceUID','SeriesInstanceUID']].sample(40,random_state=1).values):
    pp=f"{D}/train_series/{st}/{se}"; f=sorted(os.listdir(pp)); d=pydicom.dcmread(pp+'/'+f[len(f)//2]); a=d.pixel_array; n+=1
    if i<8: print(a.shape, d.file_meta.TransferSyntaxUID.name, getattr(d,'SliceThickness',None), getattr(d,'PixelSpacing',None))
print('sec per slice', (time.time()-t)/n)
print(d)


In [ ]:
# duplicate check: per-study plane coverage
g=ts.groupby('StudyInstanceUID').Anatomical_Plane.agg(lambda x: ''.join(sorted(set(v[0] for v in x))))
print(g.value_counts().head())
fsat=ts[(ts.Fluid_Sensitive==1)].groupby('StudyInstanceUID').Anatomical_Plane.agg(lambda x: ''.join(sorted(set(v[0] for v in x))))
print(fsat.value_counts().head())
import subprocess; print(subprocess.run('nvidia-smi; df -h /kaggle/working; nproc; free -g',shell=True,capture_output=True,text=True).stdout)
